# Ingesta de compañías productoras (incremental)

Lee la carpeta `production_company` (CSV) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.production_companies` con `MERGE` (llave `company_id` + `file_date`), particionada por `file_date`.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del folder production_company

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

### Lectura del archivo

Primero lee la carpeta `production_company` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
production_companies_df = (
    spark.read
        .format("csv")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_company")
)

In [0]:
production_companies_df_shema = StructType([
    StructField("companyId", IntegerType(), True),
    StructField("companyName", StringType(), True)
])
production_companies_df = (
    spark.read
        .format('csv')
        .schema(production_companies_df_shema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_company")
)

### Transformación de columnas

Renombra `companyId` y `companyName` a snake_case y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
production_companies_df = (
    add_ingestion_date(production_companies_df)
    .select(
        col("companyId").alias("company_id"),
        col("companyName").alias("company_name"),
        lit(v_env).alias("env"),
        lit(v_file_date).alias("file_date")
    )
)

### Escritura en Silver

Las celdas comentadas conservan estrategias anteriores (borrar partición + `append`). La carga activa es `merge_delta_lake` sobre `movie_silver.production_companies` con la llave `company_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "production_companies", "file_date", v_file_date)

In [0]:
"""
(
    production_companies_df.write
    .mode("append")
    .partitionBy("file_date")
    .saveAsTable("movie_silver.production_companies")
)
"""

In [0]:
merge_condition = 'trgt.company_id = src.company_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = production_companies_df, schema_name = "movie_silver", table_name = "production_companies", merge_condition = merge_condition, partition_column = "file_date")

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")